# ALL Met Painting Eyes

## The Met API

https://metmuseum.github.io/

https://github.com/metmuseum/openaccess

**Please limit request rate to 80 requests per second.**

In [ ]:
from google.colab import userdata
PAT_XYZ = userdata.get("PAT_XYZ")

In [ ]:
!pip install mediapipe
!pip install ultralytics
!wget https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task
!mkdir json && wget -P json https://raw.githubusercontent.com/acervos-digitais/met-faces-utils/refs/heads/main/json/mp_masks_definitions.json
!wget https://raw.githubusercontent.com/acervos-digitais/met-faces-utils/refs/heads/main/utils.py
!wget https://raw.githubusercontent.com/acervos-digitais/met-faces-utils/refs/heads/main/utils_paintings.py
!git clone https://{PAT_XYZ}@github.com/acervos-digitais/met-faces-data.git data
!cd data && git config user.name "Thiago Hersan" && git config user.email "thiago.hersan+github@gmail.com"

In [ ]:
from utils import get_combined_jsons

from utils_paintings import PaintingsUtils

DATA_DIR = "./data"

JSON_DIR = f"{DATA_DIR}/json"
IMG_DIR = f"{DATA_DIR}/image"

## Painting Objects

- $15\text{,}178$ on website
- $15\text{,}050$ (±100) available in API
- $14\text{,}233$ (±50) have images (according to `hasImages=true` API query param)
- $9\text{,}015$ ($63\%$) actually have images that can be downloaded
- $5\text{,}000$ ($55\%$) have faces
- $4\text{,}498$ ($90\%$) have extractable eyes
- $3\text{,}640$ ($80\%$) are from color images

In [ ]:
obj_ids = PaintingsUtils.get_object_ids()
len(obj_ids)

## Get Object Metadata

In [ ]:
mPU = PaintingsUtils(JSON_DIR, IMG_DIR)

for cnt,oid in enumerate(obj_ids):
  if cnt % 16 == 0:
    print(f"{cnt} / {len(obj_ids)}")

  mPU.get_obj_data(oid)

## Get Faces, Landmarks and Eyes

In [ ]:
mPU = PaintingsUtils(JSON_DIR, IMG_DIR, with_detectors=True)
objects_data = get_combined_jsons(mPU.json_objs_dir)
len(objects_data)

In [ ]:
for cnt,obj_data in enumerate(objects_data):
  if cnt % 16 == 0:
    print(f"{cnt} / {len(objects_data)}")

  if mPU.is_done(obj_data):
    continue

  img = mPU.get_image(obj_data["primaryImage"])
  if img is None:
    continue

  face_data = mPU.get_face_data(obj_data, img)
  if face_data is None:
    continue

  landmark_data = mPU.get_landmark_data(face_data, img)
  if landmark_data is None:
    continue

  mPU.get_eye_images(landmark_data, img)

## Checks

In [ ]:
mPU = PaintingsUtils(JSON_DIR, IMG_DIR)
objs = get_combined_jsons(mPU.json_objs_dir)
faces = get_combined_jsons(mPU.json_faces_dir)
landmarks = get_combined_jsons(mPU.json_landmarks_dir)

no_imgs_s = set(mPU.no_imgs)
ye_imgs_s = set([o["objectID"] for o in objs])

no_faces_s = set(mPU.no_faces)
ye_faces_s = set([o["objectID"] for o in faces])

no_lands_s = set(mPU.no_landmarks)
ye_lands_s = set([o["objectID"] for o in landmarks])

print(len(no_imgs_s), len(ye_imgs_s), len(no_imgs_s) + len(ye_imgs_s))
print(len(no_faces_s), len(ye_faces_s), len(no_faces_s) + len(ye_faces_s))
print(len(no_lands_s), len(ye_lands_s), len(no_lands_s) + len(ye_lands_s))

print(len(no_imgs_s.intersection(ye_imgs_s)))
print(len(no_faces_s.intersection(ye_faces_s)))
print(len(no_lands_s.intersection(ye_lands_s)))

## Filter B&W

In [ ]:
import json
from os import listdir
from PIL import Image as PImage, ImageStat as PImageStat

from utils import get_combined_jsons
from utils_paintings import PaintingsUtils

DATA_DIR = "./data"
JSON_DIR = f"{DATA_DIR}/json"
IMG_DIR = f"{DATA_DIR}/image"

mPU = PaintingsUtils(JSON_DIR, IMG_DIR)
landmarks_data = get_combined_jsons(mPU.json_landmarks_dir)
len(landmarks_data)

In [ ]:
no_color = []
eye_imgs_filenames = listdir(mPU.image_eyes_dir)

for ocnt,obj_data in enumerate(landmarks_data):
  oid = obj_data["objectID"]
  eye_filenames = [f for f in eye_imgs_filenames if f.startswith(f"{oid}_")]

  if len(eye_filenames) < 1:
    print("NO FILES:", oid)

  no_color_cnt = 0
  for ecnt,fname in enumerate(eye_filenames):
    img = PImage.open(f"{mPU.image_eyes_dir}/{fname}")
    stat = PImageStat.Stat(img)

    regeb = (stat.sum[0] == stat.sum[1]) and (stat.sum[1] == stat.sum[2])

    if regeb:
      no_color_cnt += 1

  if no_color_cnt == len(eye_filenames):
    no_color.append(oid)
  elif no_color_cnt > 0:
    print("OOH OHH UNSURE:", oid)
    no_color.append(oid)

len(no_color)

In [ ]:
with open(f"{JSON_DIR}/no_color.json", "w") as ofp:
  json.dump(no_color, ofp, ensure_ascii=False)

In [ ]:
import json

from os import listdir, makedirs
from shutil import copy2

from utils_paintings import PaintingsUtils

DATA_DIR = "./data"
JSON_DIR = f"{DATA_DIR}/json"
IMG_DIR = f"{DATA_DIR}/image"

JSON_COLOR_DIR = f"{JSON_DIR}/landmarks-color"
IMG_COLOR_DIR = f"{IMG_DIR}/eyes-color"

makedirs(JSON_COLOR_DIR, exist_ok=True)
makedirs(IMG_COLOR_DIR, exist_ok=True)

with open(f"{JSON_DIR}/no_color.json", "r") as ifp:
  no_color = json.load(ifp)

mPU = PaintingsUtils(JSON_DIR, IMG_DIR)

json_to_copy = [f for f in listdir(mPU.json_landmarks_dir) if f.endswith("json") and int(f.split(".")[0]) not in no_color]
img_to_copy = [f for f in listdir(mPU.image_eyes_dir) if f.endswith("avif") and int(f.split("_")[0]) not in no_color]

for f in json_to_copy:
  copy2(f"{mPU.json_landmarks_dir}/{f}", f"{JSON_COLOR_DIR}/{f}")
  
for f in img_to_copy:
  copy2(f"{mPU.image_eyes_dir}/{f}", f"{IMG_COLOR_DIR}/{f}")

## Export csv

In [ ]:
from utils_paintings import PaintingsUtils

DATA_DIR = "./data"

CSV_DIR = f"{DATA_DIR}/csv"
JSON_DIR = f"{DATA_DIR}/json"
IMG_DIR = f"{DATA_DIR}/image"

mPU = PaintingsUtils(JSON_DIR, IMG_DIR)

mPU.export_csv(mPU.json_objs_dir, f"{CSV_DIR}/objects.csv")
mPU.export_csv(mPU.json_faces_dir, f"{CSV_DIR}/faces.csv")
mPU.export_csv(mPU.json_landmarks_dir, f"{CSV_DIR}/landmarks.csv")
mPU.export_csv(f"{JSON_DIR}/landmarks-color", f"{CSV_DIR}/landmarks-color.csv")